# Subgroup Meta-Analysis: Human-AI Collaboration
## Chia nhóm theo Moderators để so sánh chi tiết

**Mục tiêu:**
- Phân tích theo từng nhóm (Industry, Task Type, AI Type, etc.)
- So sánh effect sizes giữa các nhóm
- Tìm hiểu moderators ảnh hưởng đến hiệu quả Human-AI collaboration

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs('Figures', exist_ok=True)
os.makedirs('Data', exist_ok=True)

# Style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Setup complete!")

## 2. Core Functions

In [ ]:
def compute_hedges_g(m1, m2, sd1, sd2, n1, n2):
    """Tính Hedges' g"""
    if sd1 <= 0 or sd2 <= 0 or n1 <= 1 or n2 <= 1:
        return np.nan, np.nan
    sd_pooled = np.sqrt(((n1 - 1) * sd1**2 + (n2 - 1) * sd2**2) / (n1 + n2 - 2))
    if sd_pooled == 0:
        return np.nan, np.nan
    d = (m1 - m2) / sd_pooled
    df = n1 + n2 - 2
    j = 1 - (3 / (4 * df - 1))
    g = j * d
    var_g = ((n1 + n2) / (n1 * n2) + (g**2) / (2 * (n1 + n2))) * (j**2)
    return g, var_g


def compute_all_effect_sizes(df):
    """Tính effect sizes cho tất cả rows"""
    n = len(df)
    es_s, var_s = np.zeros(n), np.zeros(n)
    es_h, var_h = np.zeros(n), np.zeros(n)
    es_a, var_a = np.zeros(n), np.zeros(n)

    for i in range(n):
        row = df.iloc[i]
        try:
            # Strong synergy
            if pd.notna(row['Avg_Perf_HumanAI_Adj']) and pd.notna(row['Avg_Perf_Baseline_Adj']):
                if row['Sd_Perf_HumanAI'] > 0 and row['Sd_Perf_Baseline'] > 0:
                    es_s[i], var_s[i] = compute_hedges_g(
                        row['Avg_Perf_HumanAI_Adj'], row['Avg_Perf_Baseline_Adj'],
                        row['Sd_Perf_HumanAI'], row['Sd_Perf_Baseline'],
                        row['N_HumanAI'], row['N_Human'])
                else:
                    es_s[i], var_s[i] = np.nan, np.nan
            else:
                es_s[i], var_s[i] = np.nan, np.nan

            # Human augmentation
            if pd.notna(row['Avg_Perf_HumanAI_Adj']) and pd.notna(row['Avg_Perf_Human_Adj']):
                if row['Sd_Perf_HumanAI'] > 0 and row['Sd_Perf_Human'] > 0:
                    es_h[i], var_h[i] = compute_hedges_g(
                        row['Avg_Perf_HumanAI_Adj'], row['Avg_Perf_Human_Adj'],
                        row['Sd_Perf_HumanAI'], row['Sd_Perf_Human'],
                        row['N_HumanAI'], row['N_HumanAI'])
                else:
                    es_h[i], var_h[i] = np.nan, np.nan
            else:
                es_h[i], var_h[i] = np.nan, np.nan

            # AI augmentation
            if pd.notna(row['Avg_Perf_HumanAI_Adj']) and pd.notna(row['Avg_Perf_AI_Adj']):
                if row['Sd_Perf_HumanAI'] > 0 and row['Sd_Perf_AI'] > 0:
                    es_a[i], var_a[i] = compute_hedges_g(
                        row['Avg_Perf_HumanAI_Adj'], row['Avg_Perf_AI_Adj'],
                        row['Sd_Perf_HumanAI'], row['Sd_Perf_AI'],
                        row['N_HumanAI'], row['N_Human'])
                else:
                    es_a[i], var_a[i] = np.nan, np.nan
            else:
                es_a[i], var_a[i] = np.nan, np.nan
        except:
            es_s[i], var_s[i] = np.nan, np.nan
            es_h[i], var_h[i] = np.nan, np.nan
            es_a[i], var_a[i] = np.nan, np.nan

    return es_s, var_s, es_h, var_h, es_a, var_a


def random_effects_meta(effect_sizes, variances):
    """Random-effects meta-analysis"""
    mask = ~np.isnan(effect_sizes) & ~np.isnan(variances) & (variances > 0)
    es = effect_sizes[mask]
    var = variances[mask]
    k = len(es)
    if k < 2:
        return None
    
    w = 1 / var
    fe_estimate = np.sum(w * es) / np.sum(w)
    Q = np.sum(w * (es - fe_estimate)**2)
    df = k - 1
    Q_pval = 1 - stats.chi2.cdf(Q, df)
    C = np.sum(w) - np.sum(w**2) / np.sum(w)
    tau2 = max(0, (Q - df) / C)
    w_re = 1 / (var + tau2)
    re_estimate = np.sum(w_re * es) / np.sum(w_re)
    se = np.sqrt(1 / np.sum(w_re))
    ci_lower = re_estimate - 1.96 * se
    ci_upper = re_estimate + 1.96 * se
    z = re_estimate / se
    p_value = 2 * (1 - norm.cdf(abs(z)))
    I2 = 100 * (Q - df) / Q if Q > df else 0
    
    return {
        'k': k, 'g': re_estimate, 'se': se,
        'ci_lower': ci_lower, 'ci_upper': ci_upper,
        'z': z, 'p': p_value, 'tau2': tau2, 'I2': I2,
        'Q': Q, 'Q_df': df, 'Q_p': Q_pval
    }


def get_significance(p):
    """Trả về significance marker"""
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    return ''


print("Functions defined!")

## 3. Load Data & Compute Effect Sizes

In [ ]:
# Load data
df = pd.read_excel("Data_Extraction_communication_public.xlsx")

print(f"Loaded {len(df)} effect sizes from {df['Paper_ID'].nunique()} papers")

# Compute effect sizes
es_s, var_s, es_h, var_h, es_a, var_a = compute_all_effect_sizes(df)

df['es_s'], df['var_s'] = es_s, var_s  # Strong synergy
df['es_h'], df['var_h'] = es_h, var_h  # Human augmentation
df['es_a'], df['var_a'] = es_a, var_a  # AI augmentation

print(f"\nEffect sizes computed:")
print(f"  Human Augmentation: {np.sum(~np.isnan(es_h))} valid")
print(f"  AI Augmentation: {np.sum(~np.isnan(es_a))} valid")
print(f"  Strong Synergy: {np.sum(~np.isnan(es_s))} valid")

## 4. Overall Meta-Analysis

In [ ]:
print("="*70)
print("OVERALL META-ANALYSIS RESULTS")
print("="*70)

comparisons = [
    ('Human Augmentation (HAI vs Human)', 'es_h', 'var_h'),
    ('AI Augmentation (HAI vs AI)', 'es_a', 'var_a'),
    ('Strong Synergy (HAI vs max)', 'es_s', 'var_s')
]

overall_results = []

for name, es_col, var_col in comparisons:
    res = random_effects_meta(df[es_col].values, df[var_col].values)
    if res:
        sig = get_significance(res['p'])
        print(f"\n{name}:")
        print(f"  k = {res['k']}")
        print(f"  g = {res['g']:.3f} [{res['ci_lower']:.3f}, {res['ci_upper']:.3f}]")
        print(f"  p = {res['p']:.4f} {sig}")
        print(f"  I² = {res['I2']:.1f}%")
        
        res['Comparison'] = name
        overall_results.append(res)

## 5. Subgroup Analysis Function

In [ ]:
def subgroup_analysis(df, moderator, es_col, var_col):
    """Chạy subgroup analysis cho một moderator"""
    results = []
    
    for group in sorted(df[moderator].dropna().unique()):
        mask = df[moderator] == group
        es = df.loc[mask, es_col].values
        var = df.loc[mask, var_col].values
        
        meta = random_effects_meta(es, var)
        if meta:
            results.append({
                'Moderator': moderator,
                'Group': group,
                'k': meta['k'],
                'g': meta['g'],
                'SE': meta['se'],
                'CI_lower': meta['ci_lower'],
                'CI_upper': meta['ci_upper'],
                'p': meta['p'],
                'sig': get_significance(meta['p']),
                'I2': meta['I2']
            })
    
    return pd.DataFrame(results)


# Define moderators to analyze
MODERATORS = [
    'Industry',
    'Task_Type',
    'AI_Type_Cleaned',
    'Participant_Expert',
    'AI_Expl_Incl',
    'Task_Output_Cleaned',
    'Baseline',
    'Comp_Type'
]

print(f"Moderators to analyze: {MODERATORS}")

## 6. Run Subgroup Analysis

In [ ]:
# Store all results
all_subgroup_results = []

# Human Augmentation
print("="*70)
print("HUMAN AUGMENTATION (HAI vs Human) - Subgroup Analysis")
print("="*70)

for mod in MODERATORS:
    if mod in df.columns:
        result = subgroup_analysis(df, mod, 'es_h', 'var_h')
        if len(result) > 0:
            result['Comparison'] = 'Human Augmentation'
            all_subgroup_results.append(result)
            
            print(f"\n### {mod}")
            for _, row in result.iterrows():
                print(f"  {row['Group']}: g = {row['g']:.3f} [{row['CI_lower']:.3f}, {row['CI_upper']:.3f}], "
                      f"k={row['k']}, p={row['p']:.4f}{row['sig']}")

In [ ]:
# AI Augmentation
print("="*70)
print("AI AUGMENTATION (HAI vs AI) - Subgroup Analysis")
print("="*70)

for mod in MODERATORS:
    if mod in df.columns:
        result = subgroup_analysis(df, mod, 'es_a', 'var_a')
        if len(result) > 0:
            result['Comparison'] = 'AI Augmentation'
            all_subgroup_results.append(result)
            
            print(f"\n### {mod}")
            for _, row in result.iterrows():
                print(f"  {row['Group']}: g = {row['g']:.3f} [{row['CI_lower']:.3f}, {row['CI_upper']:.3f}], "
                      f"k={row['k']}, p={row['p']:.4f}{row['sig']}")

In [ ]:
# Strong Synergy
print("="*70)
print("STRONG SYNERGY (HAI vs max) - Subgroup Analysis")
print("="*70)

for mod in MODERATORS:
    if mod in df.columns:
        result = subgroup_analysis(df, mod, 'es_s', 'var_s')
        if len(result) > 0:
            result['Comparison'] = 'Strong Synergy'
            all_subgroup_results.append(result)
            
            print(f"\n### {mod}")
            for _, row in result.iterrows():
                print(f"  {row['Group']}: g = {row['g']:.3f} [{row['CI_lower']:.3f}, {row['CI_upper']:.3f}], "
                      f"k={row['k']}, p={row['p']:.4f}{row['sig']}")

In [ ]:
# Combine all results
all_results_df = pd.concat(all_subgroup_results, ignore_index=True)
print(f"\nTotal subgroup results: {len(all_results_df)}")
all_results_df.head(10)

## 7. Visualizations

In [ ]:
def plot_subgroup_by_moderator(all_results, moderator, figsize=(14, 6)):
    """Tạo bar chart so sánh effect sizes theo moderator"""
    mod_data = all_results[all_results['Moderator'] == moderator].copy()
    
    if len(mod_data) == 0:
        print(f"No data for {moderator}")
        return
    
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    comparisons = ['Human Augmentation', 'AI Augmentation', 'Strong Synergy']
    titles = ['HAI vs Human', 'HAI vs AI', 'HAI vs max(H,AI)']
    
    for idx, (comp, title) in enumerate(zip(comparisons, titles)):
        ax = axes[idx]
        data = mod_data[mod_data['Comparison'] == comp].copy()
        
        if len(data) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title)
            continue
        
        # Sort by effect size
        data = data.sort_values('g', ascending=True)
        
        groups = data['Group'].values
        effects = data['g'].values
        errors = [data['g'] - data['CI_lower'], data['CI_upper'] - data['g']]
        
        # Colors
        colors = ['darkgreen' if (e > 0 and p < 0.05) else
                  'darkred' if (e < 0 and p < 0.05) else 'gray'
                  for e, p in zip(effects, data['p'].values)]
        
        # Bar chart
        bars = ax.barh(range(len(groups)), effects, xerr=errors,
                       color=colors, capsize=4, alpha=0.7, edgecolor='black')
        
        ax.axvline(0, color='black', linestyle='--', linewidth=1)
        ax.set_yticks(range(len(groups)))
        ax.set_yticklabels([f"{g} (k={k})" for g, k in zip(groups, data['k'].values)])
        ax.set_xlabel("Hedges' g")
        ax.set_title(title, fontweight='bold')
        
        # Add significance markers
        for i, (_, row) in enumerate(data.iterrows()):
            if row['sig']:
                x_pos = row['g'] + 0.05 if row['g'] > 0 else row['g'] - 0.1
                ax.text(x_pos, i, row['sig'], va='center', fontsize=10, fontweight='bold')
    
    plt.suptitle(f"Subgroup Analysis by {moderator}", fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'Figures/Subgroup_{moderator}.png', dpi=300, bbox_inches='tight')
    plt.show()


# Plot for each moderator
for mod in MODERATORS:
    if mod in all_results_df['Moderator'].values:
        plot_subgroup_by_moderator(all_results_df, mod)

## 8. Summary Heatmap

In [ ]:
# Create heatmap data
heatmap_data = all_results_df.pivot_table(
    index=['Moderator', 'Group'],
    columns='Comparison',
    values='g',
    aggfunc='first'
)

# Reorder columns
col_order = ['Human Augmentation', 'AI Augmentation', 'Strong Synergy']
heatmap_data = heatmap_data[[c for c in col_order if c in heatmap_data.columns]]

# Plot
fig, ax = plt.subplots(figsize=(10, max(8, len(heatmap_data) * 0.35)))

sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            cbar_kws={'label': "Hedges' g"})

ax.set_title('Summary: Effect Sizes by Subgroups', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig('Figures/Subgroup_Heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Key Findings Table

In [ ]:
# Significant findings only
significant = all_results_df[all_results_df['p'] < 0.05].copy()
significant = significant.sort_values(['Comparison', 'Moderator', 'g'], ascending=[True, True, False])

print("="*70)
print("SIGNIFICANT FINDINGS (p < 0.05)")
print("="*70)

for comp in ['Human Augmentation', 'AI Augmentation', 'Strong Synergy']:
    comp_data = significant[significant['Comparison'] == comp]
    if len(comp_data) > 0:
        print(f"\n{comp}:")
        print("-" * 60)
        for _, row in comp_data.iterrows():
            direction = "↑" if row['g'] > 0 else "↓"
            print(f"  {row['Moderator']}: {row['Group']} → g = {row['g']:.3f} {direction} {row['sig']}")

## 10. Save Results

In [ ]:
# Save all results
all_results_df.to_csv('Data/Subgroup_All_Results.csv', index=False)

# Save by comparison
for comp in ['Human Augmentation', 'AI Augmentation', 'Strong Synergy']:
    comp_data = all_results_df[all_results_df['Comparison'] == comp]
    filename = f"Data/Subgroup_{comp.replace(' ', '_')}.csv"
    comp_data.to_csv(filename, index=False)
    print(f"Saved: {filename}")

# Save pivot table
heatmap_data.to_csv('Data/Subgroup_Summary_Pivot.csv')

# Save data with effect sizes
df.to_csv('Data/Data_with_EffectSizes.csv', index=False)

print("\nAll files saved!")

## 11. Summary Report

In [ ]:
print("="*70)
print("SUMMARY REPORT")
print("="*70)

print("\n📊 KEY INSIGHTS:")
print()

# Industry insights
print("1. BY INDUSTRY:")
industry_data = all_results_df[all_results_df['Moderator'] == 'Industry']
for _, row in industry_data[industry_data['p'] < 0.05].iterrows():
    print(f"   • {row['Group']} ({row['Comparison']}): g = {row['g']:.2f}{row['sig']}")

# Task type insights
print("\n2. BY TASK TYPE:")
task_data = all_results_df[all_results_df['Moderator'] == 'Task_Type']
for _, row in task_data[task_data['p'] < 0.05].iterrows():
    print(f"   • {row['Group']} ({row['Comparison']}): g = {row['g']:.2f}{row['sig']}")

# Baseline insights
print("\n3. BY BASELINE (AI vs Human quality):")
baseline_data = all_results_df[all_results_df['Moderator'] == 'Baseline']
for _, row in baseline_data[baseline_data['p'] < 0.05].iterrows():
    print(f"   • {row['Group']} ({row['Comparison']}): g = {row['g']:.2f}{row['sig']}")

# Expert insights
print("\n4. BY EXPERTISE:")
expert_data = all_results_df[all_results_df['Moderator'] == 'Participant_Expert']
for _, row in expert_data[expert_data['p'] < 0.05].iterrows():
    expert_label = "Expert" if row['Group'] == 'Yes' else "Non-Expert"
    print(f"   • {expert_label} ({row['Comparison']}): g = {row['g']:.2f}{row['sig']}")

print("\n" + "="*70)